# Determinant from Conjugate Gradient

I want to test the idea in this Stackflow [post](https://scicomp.stackexchange.com/questions/4883/calculating-determinant-while-solving-ax-b-using-cg).

We modify the Julia code from [Algorithms from the Book](https://github.com/KennethLange/AlgorithmsFromTheBook.jl/blob/master/src/conjugategradients.jl).

In [51]:
using LinearAlgebra, SparseArrays

""" 
Solves the equation A*x = b by the conjugate gradient method. 
The matrix A is assumed positive definite; tol is tolerance for 
testing convergence.
"""
function conjugategradients(
        A   :: SparseMatrixCSC{T, Int},
        b   :: Vector{T}, 
        tol :: T
        ) where T <: Real
    n = size(A, 1)
    
    if n == 1
        return (b / A[1, 1], log(A[1, 1]))
    else
        x = zeros(T, n) # solution vector starts at 0
        v = copy(b) # search direction
        r = copy(b) # residual b - A * x
        s = sum(abs2, r) # residual sum of squares
        logdt = zero(T) # estimate of log(determinant)
        for j = 1:n # perform the conjugate gradient steps 
            av = A * v
            c = dot(v, av)
            # println("c = $c")
            if c <= zero(T)
                return (x, logdt)
            end
            t = s / c # step length
            # println("t = $t")
            logdt -= log(t)
            x = x + t * v
            r = r - t * av
            d = sum(abs2, r)
            #if sqrt(d) < tol # convergence test
            #    return (x, logdt)
            #end
            v = r + (d / s) * v
            s = d
        end
        return (x, logdt)
    end
end

conjugategradients

Test example:

In [58]:
using Random
#Random.seed!(123)

(n, tol) = (100, 1e-15)
A = sprandn(n, n, .01)
rho = opnorm(A, 1)
A = 0.5 * (A + A') + rho * I
x = randn(n)
b = A * x
y, ld = conjugategradients(A, b, tol)
@show norm(x - y)
@show ld, logdet(A)

norm(x - y) = 2.4760040639807405e-15
(ld, logdet(A)) = (155.66600987167348, 156.4018094246985)


(155.66600987167348, 156.4018094246985)